<a href="https://colab.research.google.com/github/kenleefk-edu/C3669C-2026-05/blob/main/Google_Colab_Narration_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
import os
import re
import asyncio

# Programmatically handle the edge-tts installation for Google Colab environments
try:
    import edge_tts
except ImportError:
    print("[+] Installing edge-tts inside Google Colab environment...")
    # Using IPython system call to ensure clean installation in Colab
    get_ipython().system("pip install edge-tts")
    import edge_tts

# Configuration
INPUT_FILE = "copilot_narration_script.md"
OUTPUT_FILE = "product_copilot_narration.mp3"

# Recommended Premium Male Voices:
# - 'en-US-BrianNeural' (Warm, articulate corporate presenter - Recommended)
# - 'en-US-GuyNeural' (Natural, classic American male speaking tone)
VOICE = "en-US-BrianNeural"
RATE = "+0%"

def extract_narration_text(filepath):
    """
    Parses the Markdown script file and strictly extracts only the spoken text
    contained within double quotation marks. This completely bypasses headers,
    slide markers, and document metadata.
    """
    if not os.path.exists(filepath):
        print(f"[-] Error: Could not find '{filepath}' in the Colab directory.")
        print("Please upload 'copilot_narration_script.md' to your Colab session files before running.")
        return None

    print(f"[+] Reading and parsing: {filepath}...")
    narration_segments = []

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            # Strictly pull text block inside quotation marks
            matches = re.findall(r'"([^"]+)"', line)
            for match in matches:
                narration_segments.append(match.strip())

    full_text = "  ".join(narration_segments)
    word_count = len(full_text.split())
    print(f"[+] Successfully extracted {word_count} words of pure narration text (all formatting skipped!).")
    return full_text

async def run_text_to_speech():
    narration_text = extract_narration_text(INPUT_FILE)
    if not narration_text:
        return

    print(f"[+] Synthesising speech using premium male voice: '{VOICE}' (speed: {RATE})...")
    communicate = edge_tts.Communicate(narration_text, VOICE, rate=RATE)

    print(f"[+] Writing high-quality audio file to: '{OUTPUT_FILE}'...")
    await communicate.save(OUTPUT_FILE)

    full_path = os.path.abspath(OUTPUT_FILE)
    print("\n" + "="*60)
    print("SUCCESS: Your presentation narration track has been generated!")
    print(f"Saved file path: {full_path}")
    print("="*60 + "\n")

    # Render an interactive audio player inside Google Colab
    try:
        from IPython.display import Audio, display
        print("[+] Displaying audio playback controls below:")
        display(Audio(OUTPUT_FILE))
    except Exception as e:
        print(f"[-] Could not render Colab audio player widget: {e}")

# Resolving IPython / Google Colab nested event loop restrictions
if __name__ == "__main__":
    if not os.path.exists(INPUT_FILE):
        print(f"[-] Wait! Please upload your narrative text as '{INPUT_FILE}' to your Colab workspace first.")
    else:
        try:
            # Check if an event loop is already running (standard in Google Colab / Jupyter)
            loop = asyncio.get_running_loop()
            if loop.is_running():
                print("[+] Active Jupyter event loop detected. Scheduling task on current loop...")
                loop.create_task(run_text_to_speech())
        except RuntimeError:
            # Fallback to standard execution if run outside an active event loop
            asyncio.run(run_text_to_speech())

[+] Active Jupyter event loop detected. Scheduling task on current loop...
